# One of four ingredients carries it

The strongest public single model on this board is a LightGBM that stacks four
things on top of an ordinary feature set: dual target encoding, target means
mapped in from the original 10,000-row dataset, four hand-written flags on
income and environmental concern, and a slow wide training schedule. It is
[najiama's notebook](https://www.kaggle.com/code/najiama/pure-lgbm-model-cv-0-94587-lb-0-94612),
and the recipe works. Credit for every ingredient below belongs there.

What the notebook does not say is **which of the four is doing the work**, because
the four arrive together. So I took them apart and measured them one at a time on
one frozen fold partition, with the model held fixed so a feature effect could not
be confused with a schedule effect.

The answer is that **one ingredient carries essentially all of it**. Dual target
encoding is worth **+0.00112** in out-of-fold AUC, winning all five folds, with a
paired fold-level standard deviation of 0.00009 — a gain twelve times its own
noise. The original-dataset means **cost** 0.00005. The four flags are worth
0.00002. Both of those are inside the noise floor, and one of them is on the wrong
side of zero.

The ladder here reproduces that measurement from scratch, and the submission cell
writes the winning arm. On my own run the winning arm scored **0.94585** on the
public leaderboard, up from 0.94450 for the same pipeline without target encoding.

Everything is measured on `StratifiedKFold(5, shuffle=True, random_state=42)` over
`train.csv` in file order. Every arm sees exactly those folds, so every difference
below is paired and every fold-level standard deviation is real.

**One warning before you read any of the leaderboard numbers here, including the
one in the title.** The public board is 20% of the test set, about 57,300 rows. At
an AUC near 0.946 with a 17.5% positive rate, the standard error of that estimate
is about **0.0016**, and that is a lower bound. The entire gain measured below is
**0.0012**. So the public board *cannot resolve this result* — my own submissions
moved 0.94450 to 0.94596, which is 0.84 standard errors and not a distinguishable
difference. What establishes the finding is the fold ladder, on 668,665 rows, paired,
where the gain is twelve times its own fold-level noise. If you fork this and your
leaderboard score wobbles by a thousandth, that is the board, not your model.

## The cell you edit

In [ ]:
CFG = {
    # The ladder. Turn any rung off to see what it was worth.
    "USE_DIGITS":        True,   # floor(x / 10**k) mod 10, k in -4..3 - generator artefacts
    "USE_INTERACTIONS":  True,   # 8 hand pairings from evgendvorkin's public notebook
    "USE_FREQUENCY":     True,   # value frequency over train+test (transductive, label-free)
    "USE_TARGET_ENC":    True,   # <- THE ONE THAT PAYS. dual sklearn TargetEncoder
    "USE_ORIGINAL":      False,  # target means from the real 10k dataset. Measured: -0.00005
    "USE_FLAGS":         False,  # the four hand flags.                    Measured: +0.00002

    # Target encoding detail. Two encoders, two smoothings, fit inside each outer
    # fold only. cv=5 is the INNER cross-fit, so no row encodes itself.
    "TE_SMOOTHINGS":     ["auto", 10.0],
    "TE_INNER_CV":       5,

    "N_SPLITS":          5,
    "SEED":              42,
    "RUN_LADDER":        True,   # False = skip straight to the submission, ~5x faster
    "LGB": dict(objective="binary", n_estimators=3000, learning_rate=0.03, num_leaves=63,
                min_child_samples=40, subsample=0.9, subsample_freq=1, colsample_bytree=0.8,
                reg_lambda=1.0, max_bin=255, verbose=-1),
    "EARLY_STOPPING":    100,
    "TARGET":            "Will_Buy_EV",
    "ID_COL":            "id",
    "DIGIT_POSITIONS":   list(range(-4, 4)),
}

## Loading, and finding the columns without naming them

Nothing below hard-codes a column list except the eight interaction pairings,
which are borrowed and named as such. Everything else is detected, so if you fork
this onto another tabular frame the ladder still runs.

In [ ]:
import os, gc, time, warnings, json
import numpy as np, pandas as pd
import inspect
import lightgbm as lgb
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import TargetEncoder
warnings.filterwarnings("ignore")
T0 = time.time()

# LightGBM renamed fit()'s validation arguments, and sklearn deprecated
# TargetEncoder's. Both are resolved from the installed signature rather than
# pinned, so this notebook runs on an older or a newer image without edits.
_LGB_FIT = inspect.signature(lgb.LGBMClassifier.fit).parameters
_TE_ARGS = inspect.signature(TargetEncoder.__init__).parameters

def fit_es(model, Xtr, ytr, Xva, yva, rounds):
    kw = dict(eval_metric="auc", callbacks=[lgb.early_stopping(rounds, verbose=False)])
    if "eval_X" in _LGB_FIT:
        model.fit(Xtr, ytr, eval_X=Xva, eval_y=yva, **kw)
    else:
        model.fit(Xtr, ytr, eval_set=[(Xva, yva)], **kw)
    return model

def make_te(smooth, cv, seed):
    kw = dict(target_type="binary", smooth=smooth, cv=cv)
    if "random_state" in _TE_ARGS:
        kw["random_state"] = seed
    return TargetEncoder(**kw)

def find(name):
    for root, _, files in os.walk("/kaggle/input"):
        if name in files:
            return os.path.join(root, name)
    raise FileNotFoundError(name)

train  = pd.read_csv(find("train.csv"))
test   = pd.read_csv(find("test.csv"))
sample = pd.read_csv(find("sample_submission.csv"))
TARGET, ID = CFG["TARGET"], CFG["ID_COL"]

def to_binary(s):
    """Yes/No, True/False or 0/1 - the frame has arrived as all three."""
    if s.dtype.kind in "iufb":
        return s.astype(np.int8).to_numpy()
    v = s.astype(str).str.strip().str.lower()
    pos = "yes" if (v == "yes").any() else sorted(v.unique())[-1]
    return (v == pos).astype(np.int8).to_numpy()

y = to_binary(train[TARGET])

drop = {ID, TARGET}
NUM = [c for c in train.columns if c not in drop and pd.api.types.is_numeric_dtype(train[c])]
CAT = [c for c in train.columns if c not in drop and c not in NUM
       and train[c].nunique() < 0.9 * len(train)]
FEATS = NUM + CAT

print(f"train {train.shape}  test {test.shape}  positive rate {y.mean():.5f}")
print(f"{len(NUM)} numeric: {NUM}")
print(f"{len(CAT)} categorical: {CAT}")
print(f"missing values: train {int(train[FEATS].isna().sum().sum())}, test {int(test[FEATS].isna().sum().sum())}")

## The blocks

Four feature blocks and two extras, each built the same way for train, validation
and test so nothing can drift between them.

**Digits.** `floor(x / 10**k) mod 10` for k in -4..3. Playground frames are drawn
from a generator fitted to real data, and the generator leaves quantisation
artefacts in the low digits. Handing the model those digits directly saves it from
having to find them through splits.

**Interactions.** Eight hand pairings, taken from
[evgendvorkin's notebook](https://www.kaggle.com/code/evgendvorkin/s6e9-single-xgb-oof-auc-0-94466).

**Frequency.** How common a value is across `train` and `test` together. This is
transductive — it reads test feature values, so it could not be reproduced from a
training fold alone — but it is label-free, so it is not target leakage. Standard
practice here, stated rather than buried.

**Target encoding.** The block this notebook is about. Every numeric, categorical
and digit column is cast to a string category and encoded twice, at two
smoothings, with `sklearn.preprocessing.TargetEncoder` fitted **inside the outer
fold only** and cross-fitted five ways inside that. Two smoothings because a hard
prior and a soft prior disagree about rare values, and the model is allowed to
decide which it trusts.

In [ ]:
INTERACTIONS = [
    ("Env_x_Subsidy",        "mul",  "Environmental_Concern_Level", "Subsidy_Available"),
    ("Env_x_Income",         "mul",  "Environmental_Concern_Level", "Annual_Income_USD"),
    ("Subsidy_x_Income",     "mul",  "Subsidy_Available",           "Annual_Income_USD"),
    ("Env_x_Commute",        "mul",  "Environmental_Concern_Level", "Daily_Commute_km"),
    ("HomeCharge_x_Subsidy", "mul",  "Home_Charging_Possible",      "Subsidy_Available"),
    ("RangeAnx_x_Subsidy",   "mul",  "Range_Anxiety_Level",         "Subsidy_Available"),
    ("Income_per_Env",       "div1", "Annual_Income_USD",           "Environmental_Concern_Level"),
    ("Commute_per_Env",      "div1", "Daily_Commute_km",            "Environmental_Concern_Level"),
]
SPEC = [t for t in INTERACTIONS if t[2] in FEATS and t[3] in FEATS]

def digit_frame(df):
    out = {}
    for c in NUM:
        v = df[c].to_numpy(dtype="float64")
        for k in CFG["DIGIT_POSITIONS"]:
            out[f"{c}_d{k}"] = np.floor(v / (10.0 ** k)).astype("int64") % 10
    return pd.DataFrame(out, index=df.index).astype("int16")

def ordinal(train_part, others):
    """Integer codes using the TRAINING fold's vocabulary. Unseen -> -1."""
    maps = {c: {v: i for i, v in enumerate(pd.unique(train_part[c]))} for c in CAT}
    def conv(df):
        o = df[NUM].copy()
        for c in CAT:
            o[c] = df[c].map(maps[c]).fillna(-1).astype("int32")
        return o
    return [conv(train_part)] + [conv(d) for d in others]

def interaction_frame(o):
    out = {}
    for name, op, a, b in SPEC:
        x, z = o[a].astype("float64"), o[b].astype("float64")
        out[name] = x * z if op == "mul" else x / (z + 1.0)
    return pd.DataFrame(out, index=o.index).astype("float32")

def frequency_frame(frames):
    """Normalised value counts over the concatenation of every frame passed in."""
    allf = pd.concat(frames, axis=0)
    out = []
    for f in frames:
        blk = {}
        for c in allf.columns:
            vc = allf[c].value_counts(normalize=True, dropna=False)
            blk[f"{c}_freq"] = f[c].map(vc).astype("float32").fillna(0.0).values
        out.append(pd.DataFrame(blk, index=f.index))
    return out

def category_frames(train_part, others):
    """Native pandas category dtype on the training fold's vocabulary."""
    cats = {c: pd.Categorical(train_part[c]).categories for c in CAT}
    def conv(df):
        o = df[NUM].copy()
        for c in CAT:
            o[c] = pd.Categorical(df[c], categories=cats[c])
        return o
    return [conv(train_part)] + [conv(d) for d in others]

## Building one arm

`build()` takes the switches and returns the three frames. Every rung is the same
function with different switches, so the arms cannot diverge by accident.

In [ ]:
ORIG = None
for cand in ("EV_Adoption_and_Range_Anxiety_Dataset.csv",):
    try:
        ORIG = pd.read_csv(find(cand))
    except FileNotFoundError:
        pass

def build(tr_raw, va_raw, te_raw, y_tr, cfg):
    frames = category_frames(tr_raw, [va_raw, te_raw])
    raws = (tr_raw, va_raw, te_raw)

    if cfg["USE_DIGITS"]:
        frames = [pd.concat([f, digit_frame(r)], axis=1) for f, r in zip(frames, raws)]
    if cfg["USE_INTERACTIONS"] and SPEC:
        ords = ordinal(tr_raw, [va_raw, te_raw])
        frames = [pd.concat([f, interaction_frame(o)], axis=1) for f, o in zip(frames, ords)]
    if cfg["USE_FREQUENCY"]:
        freqs = frequency_frame(frames)
        frames = [pd.concat([f, q], axis=1) for f, q in zip(frames, freqs)]

    extra = ([], [], [])
    if cfg["USE_TARGET_ENC"]:
        def te_source(r):
            parts = [r[FEATS].astype(str)]
            if cfg["USE_DIGITS"]:
                parts.append(digit_frame(r).astype(str))
            o = pd.concat(parts, axis=1)
            o.columns = [f"{c}_cat" for c in o.columns]
            return o
        s_tr, s_va, s_te = (te_source(r) for r in raws)
        cols = list(s_tr.columns)
        for sm in cfg["TE_SMOOTHINGS"]:
            enc = make_te(sm, cfg["TE_INNER_CV"], cfg["SEED"])
            arrs = (enc.fit_transform(s_tr, y_tr).astype("float32"),
                    enc.transform(s_va).astype("float32"),
                    enc.transform(s_te).astype("float32"))
            names = [f"{c}_TE_{sm}" for c in cols]
            for bucket, arr, f in zip(extra, arrs, frames):
                bucket.append(pd.DataFrame(arr, columns=names, index=f.index))
            del enc, arrs; gc.collect()
        del s_tr, s_va, s_te; gc.collect()

    if cfg["USE_ORIGINAL"] and ORIG is not None:
        yo = to_binary(ORIG[TARGET])
        prior = float(yo.mean())
        maps = {c: pd.Series(yo, index=ORIG.index).groupby(ORIG[c]).mean()
                for c in FEATS if c in ORIG.columns}
        for bucket, r, f in zip(extra, raws, frames):
            bucket.append(pd.DataFrame(
                {f"{c}_org": r[c].map(m).fillna(prior).astype("float32").values
                 for c, m in maps.items()}, index=f.index))

    if cfg["USE_FLAGS"]:
        for bucket, r, f in zip(extra, raws, frames):
            inc = r["Annual_Income_USD"]
            bucket.append(pd.DataFrame({
                "is_30k_spike":         (inc == 30000.0).astype("int8").values,
                "is_millionaire_cliff": (inc >= 170537.0).astype("int8").values,
                "is_dead_zone":         inc.between(38000.0, 42000.0).astype("int8").values,
                "is_env_hater":         (r["Environmental_Concern_Level"] == 1).astype("int8").values,
            }, index=f.index))

    out = [pd.concat([f] + b, axis=1) if b else f for f, b in zip(frames, extra)]
    del frames, extra; gc.collect()
    return out

def run_arm(cfg, label):
    skf = StratifiedKFold(n_splits=cfg["N_SPLITS"], shuffle=True, random_state=cfg["SEED"])
    oof = np.zeros(len(train)); pred = np.zeros(len(test)); aucs = []
    t0 = time.time()
    for k, (i_tr, i_va) in enumerate(skf.split(train, y)):
        tr_raw, va_raw = train.iloc[i_tr], train.iloc[i_va]
        Xtr, Xva, Xte = build(tr_raw, va_raw, test, y[i_tr], cfg)
        m = fit_es(lgb.LGBMClassifier(random_state=cfg["SEED"] + k, n_jobs=-1, **cfg["LGB"]),
                   Xtr, y[i_tr], Xva, y[i_va], cfg["EARLY_STOPPING"])
        oof[i_va] = m.predict_proba(Xva)[:, 1]
        pred += m.predict_proba(Xte)[:, 1] / cfg["N_SPLITS"]
        aucs.append(roc_auc_score(y[i_va], oof[i_va]))
        if k == 0:
            print(f"  {label}: {Xtr.shape[1]} columns", flush=True)
        del Xtr, Xva, Xte, m; gc.collect()
    print(f"  {label}: OOF {roc_auc_score(y, oof):.6f}   {time.time()-t0:.0f}s", flush=True)
    return oof, pred, np.array(aucs)

## The ladder

Five arms. Each one changes exactly one switch from the arm above it, so each row
is a clean paired comparison against the row above.

In [ ]:
ARMS = [
    ("base: digits + interactions + frequency", dict(USE_TARGET_ENC=False)),
    ("+ dual target encoding",                  dict(USE_TARGET_ENC=True)),
    ("+ original-dataset means",                dict(USE_TARGET_ENC=True, USE_ORIGINAL=True)),
    ("+ four hand flags",                       dict(USE_TARGET_ENC=True, USE_ORIGINAL=True, USE_FLAGS=True)),
    ("target encoding + flags, no original",    dict(USE_TARGET_ENC=True, USE_FLAGS=True)),
]

RESULTS = {}
if CFG["RUN_LADDER"]:
    for label, over in ARMS:
        cfg = dict(CFG); cfg.update(over)
        RESULTS[label] = run_arm(cfg, label)
else:
    print("RUN_LADDER is off - skipping to the submission.")

In [ ]:
if RESULTS:
    base_aucs = RESULTS[ARMS[0][0]][2]
    rows = []
    for label, _ in ARMS:
        _, _, a = RESULTS[label]
        d = a - base_aucs
        rows.append({
            "arm": label,
            "OOF AUC": round(float(roc_auc_score(y, RESULTS[label][0])), 6),
            "fold mean": round(float(a.mean()), 6),
            "vs base": round(float(d.mean()), 6),
            "sd of delta": round(float(d.std(ddof=1)), 6) if d.std() else 0.0,
            "folds won": f"{int((d > 0).sum())}/{len(d)}",
        })
    LADDER = pd.DataFrame(rows)
    display(LADDER)

### Reading it

The gap between the first two rows is the whole story. Target encoding moves the
out-of-fold AUC by about **+0.0011**, and it wins **every fold**, with a paired
standard deviation around **0.0001**. A gain that is an order of magnitude larger
than its own fold-to-fold spread is not a lucky split.

The next two rows are the honest negatives. Mapping target means in from the
original 10,000-row dataset makes the score **worse**, which is easy to see once
you look at the coverage: the income column has 8,915 distinct values in a
10,000-row file, so a matched value carries a mean computed from roughly one
observation. That is not a prior, that is a coin flip with a decimal point.
[andrewleal70 reached the same conclusion](https://www.kaggle.com/code/andrewleal70/s6e9-i-falsified-8-leaderboard-levers)
about the same file by a completely different route, measuring it as augmentation
rather than as an encoding, and also found it costs score.

The four hand flags are worth about +0.00002, which is nothing. They are not
wrong — the income spike at 30,000 and the cliff above 170,537 are both real
generator artefacts — but the digit block has already told the model where the
round numbers are, so the flags are re-describing something it can see.

That is the useful shape of this result. **Three of the four ingredients are
re-describing information the base already carries. One of them adds information
the base does not have**, which is what the target of every row says about the
value in every column.

## Why target encoding is the one that pays

Frequency encoding and target encoding look like the same move. They are not.
Frequency asks *how often does this value occur*, which is a property of the
feature column alone. Target encoding asks *what happened to the rows that had
this value*, which needs the label. On a synthetic frame where the generator has
laid down thousands of near-duplicate values, the second question has an answer
and the first mostly does not.

The check below is cheap: correlate each block against the label directly.

In [ ]:
if RESULTS:
    from scipy.stats import spearmanr
    oof_base = RESULTS[ARMS[0][0]][0]
    oof_te   = RESULTS[ARMS[1][0]][0]
    print(f"rank correlation between the two arms' OOF: {spearmanr(oof_base, oof_te).statistic:.5f}")
    moved = np.mean(np.abs(pd.Series(oof_base).rank(pct=True)
                           - pd.Series(oof_te).rank(pct=True)) > 0.01)
    print(f"share of rows the two arms rank more than 1 percentile apart: {moved:.1%}")

## The submission

The winning arm, written out. **This exact file scored 0.94585 on the public
leaderboard**, against 0.94450 for the same pipeline with target encoding switched
off. I submitted the notebook's own output rather than a tidier version from a
private pipeline, so the number in the title is the number a fork produces.

Read that pair of numbers as *consistent with* the ladder rather than as proof of
it. The 0.0013 between them is 0.84 standard errors on a 57,300-row board. The
ladder is the evidence; the leaderboard just failed to contradict it.

In [ ]:
best_label = ARMS[1][0] if RESULTS else None
if RESULTS:
    pred = RESULTS[best_label][1]
else:
    cfg = dict(CFG)
    _, pred, aucs = run_arm(cfg, "submission arm")
    print(f"OOF fold mean {aucs.mean():.6f}")

sub = sample.copy()
idc = ID if ID in sub.columns else sub.columns[0]
sub[TARGET if TARGET in sub.columns else sub.columns[1]] = pred
sub.to_csv("submission.csv", index=False)
print(sub.head())
print(f"wrote submission.csv  {sub.shape}   total runtime {(time.time()-T0)/60:.1f} min")

## Cross-check against a published OOF library

The base arm here should be the same model as member `h` of the
[S6E9 OOF library](https://www.kaggle.com/datasets/dariushafshar/s6e9-golem-oof-library),
which holds out-of-fold and test arrays for nineteen models on **these same folds**.
If the two disagree, one of them is wrong and it is worth knowing which. If you are
building a stack, mounting that dataset saves you re-running the base entirely.

In [ ]:
try:
    lib = None
    for root, _, files in os.walk("/kaggle/input"):
        if "oof_h.npy" in files:
            lib = root; break
    if lib is None:
        print("library not mounted - skipping the cross-check")
    elif RESULTS:
        from scipy.stats import spearmanr
        oof_lib = np.load(os.path.join(lib, "oof_h.npy"))
        oof_here = RESULTS[ARMS[0][0]][0]
        if len(oof_lib) == len(oof_here):
            print(f"library member h OOF AUC : {roc_auc_score(y, oof_lib):.6f}")
            print(f"this notebook base arm   : {roc_auc_score(y, oof_here):.6f}")
            print(f"rank correlation         : {spearmanr(oof_lib, oof_here).statistic:.5f}")
        else:
            print(f"length mismatch {len(oof_lib)} vs {len(oof_here)} - different frame, skipping")
except Exception as e:
    print(f"cross-check skipped: {type(e).__name__}: {e}")

## What I tried next, and what happened

Three follow-ups, run on the same folds after this notebook was written. Two of
them were mine rather than borrowed, and both failed, so they are here rather than
quietly dropped.

**Target-encoding pairs of columns instead of single ones. Failed, twice.** If
encoding one column is worth +0.0011, encoding every pair should find the cases
where a value of A only matters when B is set. All 78 raw pairs: **−0.00016, zero
folds out of five.** The obvious diagnosis was support — `Annual_Income_USD` has
13,214 distinct values and `Daily_Commute_km` has 805, so that single pair spans
**10,637,270 possible keys over 668,665 rows**, about 0.06 rows per key, and the
inner cross-fit correctly shrinks nearly all of them to the prior. So I restricted
it to the 55 pairs whose both operands have at most 50 distinct values. That
recovered about half the damage (+0.00009, five folds out of five, against the
unrestricted version) and **still lost: −0.00007, zero folds out of five.**

The reading I did not want is the right one. A boosted tree already represents a
two-column interaction by splitting on one column and then the other. A
target-encoded cross hands it a summary it can build for itself, and charges 55
extra columns for the split search to scan every iteration. Which is also why the
eight hand-built interactions in the base block *do* help: they are products and
ratios, shapes a tree approximates badly. Mechanical crosses are not.

**The original author's exact training schedule. No better than a cheap
approximation of it.** He runs learning rate 0.005 with 40,000 trees and 1,024
bins. Running that literally scores **0.945724**; running it at 0.015 with 511 bins
scores **0.945731** — a difference of six millionths, two folds out of five, at
**2.2 times the wall clock**. The schedule is worth about +0.00011 over no schedule
at all, and you get all of it cheaply. Slow and wide saturates well before where he
set it.

**Blending. Real in cross-validation, invisible on the board.** A nested greedy
blend over twelve models beat the best single by +0.000095 with all five folds
positive. Seven of those twelve earned zero weight in every fold. Submitted, it
scored 0.94593 against the best single's 0.94596 — which proves nothing either way,
because 0.00003 is 0.02 standard errors. If you blend here, believe your folds and
ignore the board.

## What I would actually change

**Tune the smoothing instead of inheriting it.** Two smoothings, `auto` and 10,
came from the notebook I borrowed the idea from. The right value depends on how
many rows sit behind the median category, which is measurable on this frame and
which I never measured.

**Encode something the tree cannot build.** Pair crosses failed because trees
already do that. The open question is what a target encoder can represent that a
depth-5 tree cannot, and I do not have a good answer yet.

**Delete the flags and the original-dataset means.** They are switches in the
config so you can watch them fail, not because they belong in a pipeline.

The number in the title is row two of the ladder, and it is this notebook's own
submission rather than a cleaner one from somewhere else. Fork it and you should
get it, to within the board's own thousandth.

## Update, 2026-09-12 — one delta: where this number sits on today's board

Public leaderboard pulled 2026-09-12 21:50 UTC, **1,696 teams**. The number in the title,
0.94585, would enter today at about position **538 (top 32%)**. The top-10% line is
**0.94640** — +0.00055 above this notebook, which is about **0.34 of the board's own
standard error** (0.0016 on ~57,300 public rows). Board median 0.94198. Nothing in the
code changed in this version; the folds, the OOF library mount and the ladder are the ones
above. Next delta, if there is one, will be a single change measured on the same folds.